# Grand Ouest BOAMP Preprocessing & Data Engineering

## tl;dr

Executed successfully. The engineered notice layer contains `226,314` Grand Ouest BOAMP notices, `74` engineered fields, `0` missing/duplicate `idweb`, and `0` records outside 2015-2025. It produced `24,869` rule-based digital candidates and `226,299` notices usable for linkage. Missing values are preserved as explicit flags, not blindly imputed.

## Context & Methods

The input is the immutable Grand Ouest annual Parquet raw layer. The output is a notice-level engineered table used by the reference benchmark, candidate-pair generation, and linkage baselines. This notebook does not decide renewal links and does not run survival analysis.


In [1]:
from __future__ import annotations

import ast
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/boamp/grand_ouest_annual_parquet").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
RAW_PARQUET_DIR = PROJECT_ROOT / "data/raw/boamp/grand_ouest_annual_parquet"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp_grand_ouest"
METADATA_DIR = PROJECT_ROOT / "data/metadata"
OUTPUT_PATH = PROCESSED_DIR / "notices_engineered.parquet"
SUMMARY_PATH = PROCESSED_DIR / "preprocessing_quality_summary.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2015
END_YEAR = 2025
EXPECTED_ROWS = 226_314
HISTORICAL_START = pd.Timestamp("2015-01-01")
HISTORICAL_END_EXCLUSIVE = pd.Timestamp("2026-01-01")

RAW_COLUMNS = [
    "idweb", "id", "contractfolderid", "objet", "famille", "famille_libelle", "filename",
    "code_departement", "code_departement_prestation", "dateparution", "datefindiffusion", "datelimitereponse",
    "nomacheteur", "titulaire", "type_procedure", "soustype_procedure", "procedure_libelle",
    "nature", "sousnature", "nature_libelle", "descripteur_code", "descripteur_libelle", "dc",
    "type_marche", "type_avis", "source_schema", "donnees", "url_avis"
]
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw Parquet dir: {RAW_PARQUET_DIR}")
print(f"Output: {OUTPUT_PATH}")


Project root: /home/senghakrou/project-gigalis
Raw Parquet dir: /home/senghakrou/project-gigalis/data/raw/boamp/grand_ouest_annual_parquet
Output: /home/senghakrou/project-gigalis/data/processed/boamp_grand_ouest/notices_engineered.parquet


## Data

### 1. Define Normalization And Extraction Helpers


In [2]:
GRAND_OUEST_REGION_BY_DEPARTMENT = {
    "22": "Bretagne", "29": "Bretagne", "35": "Bretagne", "56": "Bretagne",
    "44": "Pays de la Loire", "49": "Pays de la Loire", "53": "Pays de la Loire", "72": "Pays de la Loire", "85": "Pays de la Loire",
    "14": "Normandie", "27": "Normandie", "50": "Normandie", "61": "Normandie", "76": "Normandie",
}
GRAND_OUEST_DEPARTMENTS = set(GRAND_OUEST_REGION_BY_DEPARTMENT)
DIGITAL_CPV_PREFIX2 = {"32", "35", "48", "72"}

TECH_KEYWORDS = {
    "cloud_hosting": ["cloud", "saas", "iaas", "paas", "hebergement", "hébergement", "datacenter", "data center", "centre de donnees", "centre de données"],
    "cybersecurity": ["cyber", "cybersecurite", "cybersécurité", "securite informatique", "sécurité informatique", "pare-feu", "firewall", "antivirus", "soc", "ssi"],
    "software": ["logiciel", "logiciels", "progiciel", "applicatif", "licence", "erp", "crm", "sirh", "ged"],
    "it_services": ["informatique", "systeme d'information", "système d'information", "infogerance", "infogérance", "maintenance informatique", "support informatique"],
    "telecom_network": ["telecommunication", "télécommunication", "telecom", "télécom", "fibre optique", "wifi", "wi-fi", "vpn", "reseau", "réseau", "connectivite", "connectivité"],
    "data_ai": ["intelligence artificielle", "machine learning", "big data", "science des donnees", "science des données", "entrepot de donnees", "entrepôt de données", "data", "donnees", "données"],
    "hardware_infrastructure": ["serveur", "stockage", "ordinateur", "poste de travail", "materiel informatique", "matériel informatique", "equipement informatique", "équipement informatique"],
}
KEYWORD_TO_SEGMENT = {kw: seg for seg, kws in TECH_KEYWORDS.items() for kw in kws}

CPV_PREFIX_TO_SEGMENT = {
    "32": "telecom_network",
    "35": "cybersecurity",
    "48": "software",
    "72": "it_services",
}

VALID_CPV_PREFIXES = {
    "03", "09", "14", "15", "16", "18", "19", "22", "24", "30", "31", "32", "33", "34", "35", "37", "38", "39",
    "41", "42", "43", "44", "45", "48", "50", "51", "55", "60", "63", "64", "65", "66", "70", "71", "72", "73",
    "75", "76", "77", "79", "80", "85", "90", "92", "98"
}

LIST_LIKE_RE = re.compile(r"^\s*\[")
CPV_RE = re.compile(r"(?<!\d)(\d{8})(?!\d)")
SIRET_CONTEXT_RE = re.compile(r"(?i)siret[^0-9]{0,40}(\d{14})")
SIREN_CONTEXT_RE = re.compile(r"(?i)siren[^0-9]{0,40}(\d{9})")
DATE_RE = re.compile(r"20\d{2}-\d{2}-\d{2}")
AMOUNT_RE = re.compile(r"(?i)(?:amount|montant|value|valeur)[^{}\[\]]{0,220}?(?:#text|text|value)?\"?\s*:?\s*\"?([0-9]{1,12}(?:[.,][0-9]{1,2})?)")
BOILERPLATE_RE = re.compile(r"\b(marche|marché|accord cadre|accord-cadre|consultation|procedure|procédure|appel d offres|appel d'offre)\b")


def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = strip_accents(str(value).lower())
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_buyer(value) -> str:
    text = normalize_text(value)
    text = re.sub(r"\b(commune de|ville de|mairie de|departement de|departement|region|conseil regional|conseil departemental|ct[eé] de cnes|communaute de communes|communaute d agglomeration|metropole)\b", " ", text)
    text = BOILERPLATE_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_list_cell(value) -> list[str]:
    if value is None or pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if LIST_LIKE_RE.match(text):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [str(item) for item in parsed if item not in [None, ""]]
        except Exception:
            return [text]
    return [text]


def normalize_department_code(value) -> str | None:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip().strip('"').strip("'").upper()
    if not text or text in {"NAN", "NONE", "NULL", "<NA>"}:
        return None
    if text in {"2A", "2B"}:
        return text
    match = re.search(r"\d+", text)
    if not match:
        return None
    number = int(match.group(0))
    return f"{number:02d}" if number < 100 else str(number)


def first_grand_ouest_department(prestation_value, publication_value) -> str | None:
    prestation = normalize_department_code(prestation_value)
    if prestation in GRAND_OUEST_DEPARTMENTS:
        return prestation
    for item in parse_list_cell(publication_value):
        department = normalize_department_code(item)
        if department in GRAND_OUEST_DEPARTMENTS:
            return department
    return None


def compact_json(values) -> str:
    return json.dumps(values, ensure_ascii=False, separators=(",", ":"))


def extract_cpv_codes(*values) -> list[str]:
    found = []
    for value in values:
        if value is None or pd.isna(value):
            continue
        for code in CPV_RE.findall(str(value)):
            if code[:2] in VALID_CPV_PREFIXES and code not in found:
                found.append(code)
    return found


def extract_identifier_candidates(value) -> tuple[list[str], list[str]]:
    text = "" if value is None or pd.isna(value) else str(value)
    sirets = []
    sirens = []
    for siret in SIRET_CONTEXT_RE.findall(text):
        if siret not in sirets:
            sirets.append(siret)
        siren = siret[:9]
        if siren not in sirens:
            sirens.append(siren)
    for siren in SIREN_CONTEXT_RE.findall(text):
        if siren not in sirens:
            sirens.append(siren)
    return sirens, sirets


def extract_amount_candidates(value) -> list[float]:
    text = "" if value is None or pd.isna(value) else str(value)
    candidates = []
    for raw in AMOUNT_RE.findall(text):
        try:
            amount = float(raw.replace(",", "."))
        except ValueError:
            continue
        if 0 <= amount <= 1_000_000_000 and amount not in candidates:
            candidates.append(amount)
        if len(candidates) >= 20:
            break
    return candidates


def extract_date_candidates(value) -> list[str]:
    text = "" if value is None or pd.isna(value) else str(value)
    seen = []
    for candidate in DATE_RE.findall(text):
        if candidate not in seen:
            seen.append(candidate)
        if len(seen) >= 30:
            break
    return seen


def keyword_hits(normalized_text: str) -> list[str]:
    hits = []
    padded = f" {normalized_text} "
    for raw_keyword, segment in KEYWORD_TO_SEGMENT.items():
        keyword = normalize_text(raw_keyword)
        if keyword and f" {keyword} " in padded:
            hits.append(raw_keyword)
    return sorted(set(hits))


def choose_segment(cpv_codes: list[str], hits: list[str]) -> tuple[str, str]:
    prefixes = [code[:2] for code in cpv_codes]
    for prefix in prefixes:
        if prefix in CPV_PREFIX_TO_SEGMENT:
            return CPV_PREFIX_TO_SEGMENT[prefix], "cpv"
    hit_segments = [KEYWORD_TO_SEGMENT[hit] for hit in hits if hit in KEYWORD_TO_SEGMENT]
    if hit_segments:
        return Counter(hit_segments).most_common(1)[0][0], "keyword"
    return "non_digital_or_uncertain", "none"


def amount_bracket(amount) -> str:
    if pd.isna(amount):
        return "missing"
    amount = float(amount)
    if amount == 0:
        return "0"
    if amount < 50_000:
        return "lt_50k"
    if amount < 200_000:
        return "50k_200k"
    if amount < 1_000_000:
        return "200k_1m"
    if amount < 5_000_000:
        return "1m_5m"
    return "ge_5m"


### 2. Transform Annual Raw Files


In [3]:
def transform_raw_frame(raw: pd.DataFrame) -> pd.DataFrame:
    engineered = pd.DataFrame(index=raw.index)
    for column in ["idweb", "id", "contractfolderid", "url_avis", "source_schema", "filename", "famille", "famille_libelle", "type_procedure", "procedure_libelle", "nature", "nature_libelle", "titulaire"]:
        engineered[column] = raw[column].astype("string") if column in raw else pd.Series(pd.NA, index=raw.index, dtype="string")

    engineered["objet_raw"] = raw.get("objet", pd.Series("", index=raw.index)).astype("string")
    engineered["buyer_name_raw"] = raw.get("nomacheteur", pd.Series("", index=raw.index)).astype("string")
    engineered["objet_normalized"] = engineered["objet_raw"].map(normalize_text).astype("string")
    engineered["buyer_name_normalized"] = engineered["buyer_name_raw"].map(normalize_buyer).astype("string")
    engineered["has_objet"] = engineered["objet_normalized"].str.len().fillna(0).gt(0)
    engineered["objet_length_chars"] = engineered["objet_raw"].fillna("").str.len().astype("int64")
    engineered["objet_length_words"] = engineered["objet_normalized"].fillna("").str.split().map(len).astype("int64")

    dateparution = pd.to_datetime(raw.get("dateparution", pd.Series(pd.NA, index=raw.index)), errors="coerce")
    engineered["dateparution"] = dateparution.dt.strftime("%Y-%m-%d").astype("string")
    engineered["datefindiffusion"] = pd.to_datetime(raw.get("datefindiffusion", pd.Series(pd.NA, index=raw.index)), errors="coerce").dt.strftime("%Y-%m-%d").astype("string")
    engineered["datelimitereponse"] = pd.to_datetime(raw.get("datelimitereponse", pd.Series(pd.NA, index=raw.index)), errors="coerce", utc=True).dt.strftime("%Y-%m-%dT%H:%M:%SZ").astype("string")
    engineered["publication_year"] = dateparution.dt.year.astype("Int64")
    engineered["publication_month"] = dateparution.dt.to_period("M").astype("string")
    engineered["publication_quarter"] = dateparution.dt.to_period("Q").astype("string")

    departments = [first_grand_ouest_department(p, d) for p, d in zip(raw.get("code_departement_prestation", []), raw.get("code_departement", []), strict=False)]
    engineered["grand_ouest_department"] = pd.Series(departments, index=raw.index, dtype="string")
    engineered["grand_ouest_region"] = engineered["grand_ouest_department"].map(GRAND_OUEST_REGION_BY_DEPARTMENT).astype("string")
    engineered["code_departement_prestation_normalized"] = raw.get("code_departement_prestation", pd.Series(pd.NA, index=raw.index)).map(normalize_department_code).astype("string")
    engineered["code_departement_publication_list_json"] = raw.get("code_departement", pd.Series(pd.NA, index=raw.index)).map(lambda value: compact_json([normalize_department_code(v) for v in parse_list_cell(value) if normalize_department_code(v)])).astype("string")

    donnees = raw.get("donnees", pd.Series("", index=raw.index)).astype("string")
    descripteur_code = raw.get("descripteur_code", pd.Series("", index=raw.index)).astype("string")
    dc = raw.get("dc", pd.Series("", index=raw.index)).astype("string")
    cpv_codes = [extract_cpv_codes(a, b, c) for a, b, c in zip(donnees, dc, descripteur_code, strict=False)]
    ids = donnees.map(extract_identifier_candidates)
    amounts = donnees.map(extract_amount_candidates)
    dates = donnees.map(extract_date_candidates)

    engineered["cpv_codes_json"] = pd.Series(cpv_codes, index=raw.index).map(compact_json).astype("string")
    engineered["cpv_prefix2_json"] = pd.Series(cpv_codes, index=raw.index).map(lambda values: compact_json(sorted({v[:2] for v in values}))).astype("string")
    engineered["primary_cpv"] = pd.Series(cpv_codes, index=raw.index).map(lambda values: values[0] if values else pd.NA).astype("string")
    engineered["primary_cpv_prefix2"] = engineered["primary_cpv"].str.slice(0, 2).astype("string")
    engineered["has_cpv"] = pd.Series(cpv_codes, index=raw.index).map(bool)

    engineered["siren_candidates_json"] = ids.map(lambda pair: compact_json(pair[0])).astype("string")
    engineered["siret_candidates_json"] = ids.map(lambda pair: compact_json(pair[1])).astype("string")
    engineered["buyer_siren_candidate"] = ids.map(lambda pair: pair[0][0] if pair[0] else pd.NA).astype("string")
    engineered["buyer_siret_candidate"] = ids.map(lambda pair: pair[1][0] if pair[1] else pd.NA).astype("string")
    engineered["has_buyer_siren"] = engineered["buyer_siren_candidate"].notna() & engineered["buyer_siren_candidate"].ne("")
    engineered["has_buyer_siret"] = engineered["buyer_siret_candidate"].notna() & engineered["buyer_siret_candidate"].ne("")
    engineered["has_buyer_name"] = engineered["buyer_name_normalized"].fillna("").str.len().gt(0)
    engineered["buyer_key_best"] = np.select(
        [engineered["has_buyer_siren"], engineered["has_buyer_siret"], engineered["has_buyer_name"]],
        [engineered["buyer_siren_candidate"], engineered["buyer_siret_candidate"].str.slice(0, 9), engineered["buyer_name_normalized"]],
        default=pd.NA,
    )
    engineered["buyer_key_best"] = pd.Series(engineered["buyer_key_best"], index=raw.index, dtype="string")
    engineered["buyer_key_source"] = np.select(
        [engineered["has_buyer_siren"], engineered["has_buyer_siret"], engineered["has_buyer_name"]],
        ["siren", "siret_to_siren", "normalized_name"],
        default="missing",
    )
    engineered["has_buyer_identifier"] = engineered["has_buyer_siren"] | engineered["has_buyer_siret"]

    combined_text = (engineered["objet_raw"].fillna("") + " " + engineered["buyer_name_raw"].fillna("") + " " + raw.get("descripteur_libelle", pd.Series("", index=raw.index)).fillna("")).map(normalize_text)
    hits = combined_text.map(keyword_hits)
    segments = [choose_segment(c, h) for c, h in zip(cpv_codes, hits, strict=False)]
    engineered["digital_keyword_hits_json"] = hits.map(compact_json).astype("string")
    engineered["digital_keyword_hit_count"] = hits.map(len).astype("int64")
    engineered["is_digital_by_keyword"] = hits.map(bool)
    engineered["is_digital_by_cpv"] = pd.Series(cpv_codes, index=raw.index).map(lambda values: bool({v[:2] for v in values} & DIGITAL_CPV_PREFIX2))
    engineered["is_digital_candidate"] = engineered["is_digital_by_keyword"] | engineered["is_digital_by_cpv"]
    engineered["technology_segment_rule_based"] = pd.Series([s[0] for s in segments], index=raw.index, dtype="string")
    engineered["technology_segment_source"] = pd.Series([s[1] for s in segments], index=raw.index, dtype="string")

    engineered["amount_candidates_json"] = amounts.map(compact_json).astype("string")
    engineered["amount_candidate_count"] = amounts.map(len).astype("int64")
    engineered["amount_best"] = amounts.map(lambda values: max(values) if values else np.nan).astype("float64")
    engineered["amount_missing"] = engineered["amount_best"].isna()
    engineered["amount_zero"] = engineered["amount_best"].fillna(-1).eq(0)
    engineered["amount_aberrant"] = engineered["amount_best"].fillna(0).gt(100_000_000)
    engineered["amount_log"] = np.log1p(engineered["amount_best"].where(engineered["amount_best"].gt(0)))
    engineered["amount_bracket"] = engineered["amount_best"].map(amount_bracket).astype("string")

    engineered["date_candidates_json"] = dates.map(compact_json).astype("string")
    engineered["contract_start_date_candidate"] = dates.map(lambda values: values[0] if values else pd.NA).astype("string")
    engineered["contract_end_date_candidate"] = dates.map(lambda values: values[-1] if len(values) >= 2 else pd.NA).astype("string")
    start_dates = pd.to_datetime(engineered["contract_start_date_candidate"], errors="coerce")
    end_dates = pd.to_datetime(engineered["contract_end_date_candidate"], errors="coerce")
    engineered["declared_duration_days"] = (end_dates - start_dates).dt.days.astype("Int64")
    engineered["declared_duration_months"] = (engineered["declared_duration_days"] / 30.4375).round(1).astype("Float64")
    engineered["duration_missing"] = engineered["declared_duration_days"].isna()
    engineered["duration_aberrant"] = engineered["declared_duration_days"].fillna(365).lt(30) | engineered["declared_duration_days"].fillna(365).gt(8 * 365)
    engineered["expected_end_date"] = end_dates.dt.strftime("%Y-%m-%d").astype("string")
    engineered["expected_end_date_source"] = np.where(engineered["expected_end_date"].notna() & engineered["expected_end_date"].ne(""), "extracted_date_candidates", "missing")

    engineered["episode_seed_key"] = engineered["contractfolderid"].where(engineered["contractfolderid"].fillna("").str.len().gt(0), engineered["idweb"]).astype("string")
    engineered["usable_for_text_similarity"] = engineered["has_objet"]
    engineered["usable_for_buyer_blocking"] = engineered["buyer_key_best"].notna() & engineered["buyer_key_best"].ne("")
    engineered["usable_for_duration_window"] = ~engineered["duration_missing"] & ~engineered["duration_aberrant"]
    engineered["usable_for_linkage"] = engineered["idweb"].fillna("").str.len().gt(0) & dateparution.notna() & engineered["usable_for_buyer_blocking"] & engineered["has_objet"]
    return engineered

frames = []
for path in sorted(RAW_PARQUET_DIR.glob("boamp_grand_ouest_*_raw.parquet")):
    raw = pd.read_parquet(path, columns=[column for column in RAW_COLUMNS if column in pq.ParquetFile(path).schema_arrow.names])
    print(f"Processing {path.name}: {len(raw):,} rows")
    frames.append(transform_raw_frame(raw))

notices = pd.concat(frames, ignore_index=True)
notices.to_parquet(OUTPUT_PATH, index=False, compression="zstd")
print(f"Wrote {len(notices):,} engineered notices to {OUTPUT_PATH}")


Processing boamp_grand_ouest_2015_raw.parquet: 18,462 rows


Processing boamp_grand_ouest_2016_raw.parquet: 20,349 rows


Processing boamp_grand_ouest_2017_raw.parquet: 20,416 rows


Processing boamp_grand_ouest_2018_raw.parquet: 20,865 rows


Processing boamp_grand_ouest_2019_raw.parquet: 21,601 rows


...

Processing boamp_grand_ouest_2022_raw.parquet: 21,349 rows


Processing boamp_grand_ouest_2023_raw.parquet: 22,012 rows


Processing boamp_grand_ouest_2024_raw.parquet: 20,507 rows


Processing boamp_grand_ouest_2025_raw.parquet: 20,754 rows


Wrote 226,314 engineered notices to /home/senghakrou/project-gigalis/data/processed/boamp_grand_ouest/notices_engineered.parquet


## Results

### 3. Validate Engineered Notice Layer


In [4]:
notices = pd.read_parquet(OUTPUT_PATH)
date_series = pd.to_datetime(notices["dateparution"], errors="coerce")
validation = {
    "rows": int(len(notices)),
    "expected_rows": EXPECTED_ROWS,
    "rows_match_expected": bool(len(notices) == EXPECTED_ROWS),
    "field_count": int(len(notices.columns)),
    "unique_idweb": int(notices["idweb"].nunique(dropna=True)),
    "missing_idweb": int(notices["idweb"].fillna("").eq("").sum()),
    "duplicate_idweb": int(notices["idweb"].duplicated().sum()),
    "invalid_dateparution": int(date_series.isna().sum()),
    "min_dateparution": date_series.min().strftime("%Y-%m-%d"),
    "max_dateparution": date_series.max().strftime("%Y-%m-%d"),
    "outside_historical_range": int(((date_series < HISTORICAL_START) | (date_series >= HISTORICAL_END_EXCLUSIVE)).sum()),
    "missing_buyer_key_best": int(notices["buyer_key_best"].fillna("").eq("").sum()),
    "missing_cpv": int((~notices["has_cpv"]).sum()),
    "digital_candidate_rows": int(notices["is_digital_candidate"].sum()),
    "usable_for_linkage_rows": int(notices["usable_for_linkage"].sum()),
}
summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_dir": str(RAW_PARQUET_DIR),
    "output_file": str(OUTPUT_PATH),
    "validation": validation,
    "rows_by_year": {str(k): int(v) for k, v in notices["publication_year"].value_counts().sort_index().items()},
    "rows_by_region": {str(k): int(v) for k, v in notices["grand_ouest_region"].value_counts().sort_index().items()},
    "digital_by_segment": {str(k): int(v) for k, v in notices.loc[notices["is_digital_candidate"], "technology_segment_rule_based"].value_counts().items()},
    "buyer_key_source_counts": {str(k): int(v) for k, v in notices["buyer_key_source"].value_counts().items()},
    "missingness": {column: int(notices[column].isna().sum() + notices[column].astype(str).str.strip().eq("").sum()) for column in ["objet_raw", "buyer_name_raw", "buyer_key_best", "primary_cpv", "amount_best", "declared_duration_days"] if column in notices},
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

checks = pd.DataFrame([
    ["Rows", f"{validation['rows']:,}"],
    ["Rows match expected", validation["rows_match_expected"]],
    ["Unique idweb", f"{validation['unique_idweb']:,}"],
    ["Missing idweb", validation["missing_idweb"]],
    ["Duplicate idweb", validation["duplicate_idweb"]],
    ["Date range", f"{validation['min_dateparution']} to {validation['max_dateparution']}"],
    ["Outside range", validation["outside_historical_range"]],
    ["Digital candidates", f"{validation['digital_candidate_rows']:,}"],
    ["Usable for linkage", f"{validation['usable_for_linkage_rows']:,}"],
], columns=["check", "value"])
display(checks)
display(pd.DataFrame(summary["rows_by_region"].items(), columns=["region", "rows"]))
display(pd.DataFrame(summary["buyer_key_source_counts"].items(), columns=["buyer_key_source", "rows"]))
assert validation["rows_match_expected"]
assert validation["missing_idweb"] == 0
assert validation["outside_historical_range"] == 0


,check,value
0,Rows,"226,314"
1,Rows match expected,True
2,Unique idweb,"226,314"
3,Missing idweb,0
4,Duplicate idweb,0
5,Date range,2015-03-02 to 2025-12-31
6,Outside range,0
7,Digital candidates,"24,869"
8,Usable for linkage,"226,299"


,region,rows
0,Bretagne,64208
1,Normandie,86180
2,Pays de la Loire,75926


,buyer_key_source,rows
0,normalized_name,196038
1,siren,30266
2,missing,10


## Takeaways

The engineered notice layer is now ready for reference benchmark preparation and candidate-pair generation. Missing values are represented as flags and source indicators rather than silently filled.
